<h1>Chapter 4 - Text Classification（文本分类）</h1>
<i>Classifying text with both representative and generative models</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)

---

This notebook is for Chapter 4 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>


If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install transformers sentence-transformers openai
# !pip install -U datasets

# **Data**

In [ ]:
from datasets import load_dataset

# Load our data
# 该数据集包含来自烂番茄（Rotten Tomatoes）的 5331 条正面的和 5331 条负面的电影评论
data = load_dataset("rotten_tomatoes")  # 使用著名的 rotten_tomatoes 数据集来训练和评估我们的模型

"""
DatasetDict({
    # 使用【训练集】来训练模型（课本和练习题（用来学习知识））
    # 用途：用于训练模型，让模型学习数据中的模式和规律
    # 作用：模型通过训练集调整内部参数，学习如何从输入特征预测输出标签
	train: Dataset({
		features: ['text', 'label'],
		num_rows: 8530
	})

	# 【验证集】（模拟考试（用来检验学习效果，调整学习方法））
	# 用途：在训练过程中评估模型性能，用于调参和模型选择
    # 作用：
    #   监控模型是否过拟合
    #   调整超参数（如学习率、网络层数等）
    #   选择最佳模型版本
	validation: Dataset({
		features: ['text', 'label'],
		num_rows: 1066
	})

	# 使用【测试集】来验证结果（正式考试（最终评估真实水平））
	# 包含 1066条未见过的数据
	# 用途：在模型训练完成后，最终评估模型的泛化能力
    # 作用：
    #   提供对模型性能的无偏估计
    #   模拟模型在真实场景中的表现
	test: Dataset({
		features: ['text', 'label'],
		num_rows: 1066
	})
})
"""
data

In [ ]:
"""

{'text': ['the rock is destined to be the 21st century\'s new " conan " and
that he\'s going to make a splash even greater than arnold schwarzenegger ,
jean-claud van damme or steven segal .',
 'things really get weird , though not particularly scary : the movie is all
portent and no content .'],
 'label': [1, 0]}

这些简短的评论被标记为正例（1）或负例（0）。这意味着我们将专注于二元情感分类

"""
data["train"][0, -1]

# **Text Classification with Representation Models（表示模型 ）**

## **Using a Task-specific Model （特定任务模型）**

In [ ]:
from transformers import pipeline

# Path to our HF model
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"  # 在推文上针对情感分析进行微调的 RoBERTa 模型

# Load model into pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

# Run inference
# 预测值
y_pred = []

"""
data["test"] 是测试集，包含 1066条未见过的数据
每条数据有两个字段：
1.text：文本内容（如电影评论），参与推理
2.label：真实标签（0=负面，1=正面），用于性能评估

pipe()推理模型根据输入文本预测文本标签
"""
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    negative_score = output[0]["score"]
    positive_score = output[2]["score"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

In [ ]:
from sklearn.metrics import classification_report


# 评估
def evaluate_performance(y_true, y_pred):
    """Create and print the classification report"""
    performance = classification_report(
        y_true, y_pred,
        target_names=["Negative Review", "Positive Review"]
    )
    print(performance)

In [ ]:
"""
完整流程：
第1步：推理预测
├─ 输入：data["test"]["text"] （测试集的文本）
├─ 模型处理：pipe() 对每个文本进行分类
└─ 输出：y_pred （预测的标签，如 [0, 1, 1, 0, ...]）

第2步：性能评估（第145行）
├─ 真实标签：data["test"]["label"] （正确答案）
├─ 预测标签：y_pred （模型预测结果）
└─ 对比两者 → 生成分类报告（精确率、召回率、F1分数）

				precision	recall	f1-score	support
Negative Review	0.76		0.88	0.81		533
Positive Review	0.86		0.72	0.78		533

accuracy 							0.80		1066
macro avg 		0.81		0.80	0.80		1066
weighted avg 	0.81		0.80	0.80		1066

"""
evaluate_performance(data["test"]["label"], y_pred)

## **Classification Tasks that Leverage Embeddings (嵌入模型)**

### Supervised Classification

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')  # 用于将文本转化为嵌入向量

# Convert text to embeddings
# 将文本转化为嵌入向量，这些嵌入向量是输入文本的数值表示
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

In [ ]:
"""
(8530, 768)
二维表示有 几个 样本，每个样本 几个 特征
即 8530 个输入文档，每个文档都有一个 768 维的嵌入向量，因此每个嵌入向量包含 768 个数值
"""
train_embeddings.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression on our train embeddings
# 基于 训练嵌入向量 和 训练标签 构建【逻辑回归模型】
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

In [ ]:
# Predict previously unseen instances
# 评估 测试数据集的嵌入向量，并预测其标签
y_pred = clf.predict(test_embeddings)

"""
完整流程：
┌─────────────────────────────────────────────────────────┐
│ 第一步：文本 → 嵌入向量（语义空间的数值表示）              │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  训练集文本 (8530条)                                     │
│  ┌──────────────┐                                       │
│  │ "great movie"│──→ model.encode() ──→ [0.1, 0.9, ...]│
│  │ "bad film"   │──→ model.encode() ──→ [0.8, 0.2, ...]│
│  │ ...          │                                       │
│  └──────────────┘                                       │
│         ↓                                               │
│  train_embeddings: (8530, 768)  ← 8530个样本×768维向量  │
│                                                          │
│  测试集文本 (1066条)                                     │
│  ┌──────────────┐                                       │
│  │ "loved it"   │──→ model.encode() ──→ [0.15, 0.85,..]│
│  │ "terrible"   │──→ model.encode() ──→ [0.75, 0.25,..]│
│  └──────────────┘                                       │
│         ↓                                               │
│  test_embeddings: (1066, 768)                           │
└─────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────┐
│ 第二步：训练分类器（学习嵌入向量与标签的关系）             │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  输入: train_embeddings + data["train"]["label"]        │
│                                                          │
│  ┌─────────────────────┐                                │
│  │ LogisticRegression  │  ← 逻辑回归分类器               │
│  │ clf.fit()           │     学习哪种向量模式对应哪个标签 │
│  └─────────────────────┘                                │
│                                                          │
│  学到的规则示例：                                          │
│  - 如果向量靠近正面区域 → label=1                        │
│  - 如果向量靠近负面区域 → label=0                        │
└─────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────┐
│ 第三步：预测测试集                                        │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  y_pred = clf.predict(test_embeddings)                  │
│                                                          │
│  测试样本              预测结果                           │
│  [0.15, 0.85, ...] ──→ 1 (正面)                         │
│  [0.75, 0.25, ...] ──→ 0 (负面)                         │
│  ...                                                    │
│                                                          │
│  y_pred = [1, 0, 1, 1, 0, ...]  ← 1066个预测标签       │
└─────────────────────────────────────────────────────────┘
                          ↓
┌─────────────────────────────────────────────────────────┐
│ 第四步：评估性能（对比预测 vs 真实）                      │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  evaluate_performance(                                   │
│      data["test"]["label"],  ← 真实答案 [1, 0, 1, ...]  │
│      y_pred                  ← 预测答案 [1, 0, 0, ...]  │
│  )                                                       │
│                                                          │
│  输出分类报告：                                           │
│  ┌─────────────────────────────────────────┐            │
│  │                precision recall f1-score │            │
│  │ Positive review    0.86    0.85    0.85  │            │
│  │ Negative review    0.85    0.86    0.85  │            │
│  │ accuracy                         0.85    │ ← 85%准确率│
│  └─────────────────────────────────────────┘            │
└─────────────────────────────────────────────────────────┘

                precision   recall      f1-score    support
Positive review 0.86        0.85        0.85        533
Negative review 0.85        0.86        0.85        533
accuracy                                0.85        1066
macro avg       0.85        0.85        0.85        1066
weighted avg    0.85        0.85        0.85        1066
"""
evaluate_performance(data["test"]["label"], y_pred)

**Tip!**  

What would happen if we would not use a classifier at all? Instead, we can average the embeddings per class and apply cosine similarity to predict which classes match the documents best:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# Average the embeddings of all documents in each target label
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values

# Find the best matching embeddings between evaluation documents and target embeddings
# 为每个文档找到最匹配的标签
sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

# Evaluate the model
evaluate_performance(data["test"]["label"], y_pred)

### Zero-shot Classification

In [ ]:
# Create embeddings for our labels
# 为标签创建嵌入向量
label_embeddings = model.encode(["A negative review", "A positive review"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
# 计算文档和标签的余弦相似度
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

**Tip!**  

What would happen if you were to use different descriptions? Use **"A very negative movie review"** and **"A very positive movie review"** to see what happens!

## **Classification with Generative Models（生成模型）**

### Encoder-decoder Models

In [ ]:
# Load our model
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device="cuda:0"
)

In [ ]:
# Prepare our data
prompt = "Is the following sentence positive or negative? "
# 遍历数据集的每一个样本，提取出文本，构建成 Prompt
data = data.map(lambda example: {"t5": prompt + example['text']})

"""
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})
"""
data

In [ ]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

### ChatGPT for Classification

In [ ]:
import openai

# Create client
client = openai.OpenAI(api_key="YOUR_KEY_HERE")

In [ ]:
def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    """Generate an output based on a prompt and an input document."""
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt.replace("[DOCUMENT]", document)
        }
    ]
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        temperature=0
    )
    return chat_completion.choices[0].message.content

In [ ]:
# Define a prompt template as a base
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# Predict the target using GPT
document = "unpretentious , charming , quirky , original"
chatgpt_generation(prompt, document)

The next step would be to run one of OpenAI's model against the entire evaluation dataset. However, only run this when you have sufficient tokens as this will call the API for the entire test dataset (1066 records).

In [ ]:
# You can skip this if you want to save your (free) credits
predictions = [chatgpt_generation(prompt, doc) for doc in tqdm(data["test"]["text"])]

In [ ]:
# Extract predictions
y_pred = [int(pred) for pred in predictions]

# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)